# 01 — Exploratory Data Analysis: IBM AML HI-Small

**Goal.** Build the structural intuition needed before feature engineering: what does the dataset look like, how imbalanced is the laundering class, what is the temporal shape, and which categorical features carry signal.

Methodology and decisions made here propagate into `src.features` (which features to engineer) and `src.evaluation` (how to set the cost matrix). Every figure is reproducible by re-running the notebook against the same `data/raw/HI-Small_Trans.csv`.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import (
    AMOUNT_PAID_COLUMN,
    LABEL_COLUMN,
    PAYMENT_FORMAT_COLUMN,
    RECEIVING_CURRENCY_COLUMN,
    TIMESTAMP_COLUMN,
    DataLoader,
)

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

In [ ]:
loader = DataLoader()
frame = loader.load()
print(f'Rows: {len(frame):,}')
print(f'Columns: {list(frame.columns)}')
frame.head(3)

## 1. Class balance

Severe class imbalance is the headline modeling constraint. The reported HI-Small positive rate is approximately 0.1%; any model evaluation that ignores this will report flattering accuracy and dangerous false-negative rates.

In [ ]:
class_balance = frame[LABEL_COLUMN].value_counts(normalize=True)
print(f'Negative rate: {class_balance.get(0, 0):.6f}')
print(f'Positive rate: {class_balance.get(1, 0):.6f}')
print(f'Implied neg/pos ratio: {class_balance.get(0, 0) / max(class_balance.get(1, 0), 1e-9):,.1f}')

## 2. Temporal distribution

Confirm transactions are time-sorted and identify the temporal range. This bounds our train/val/test split decisions in `src.data.splits`.

In [ ]:
print(f'Min timestamp: {frame[TIMESTAMP_COLUMN].min()}')
print(f'Max timestamp: {frame[TIMESTAMP_COLUMN].max()}')
print(f'Range: {frame[TIMESTAMP_COLUMN].max() - frame[TIMESTAMP_COLUMN].min()}')

daily_volume = frame.set_index(TIMESTAMP_COLUMN).resample('D').size()
fig, ax = plt.subplots(figsize=(11, 3.5))
daily_volume.plot(ax=ax, color='#0f172a', linewidth=1.0)
ax.set_title('Daily transaction volume', fontsize=12)
ax.set_ylabel('Transactions per day')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 3. Transaction amount distribution

Amount distributions in AML data are heavily right-skewed and bi-modal at structuring thresholds. We compare the distribution by class to confirm the structuring signal is visible in raw data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, ax in zip([0, 1], axes):
    subset = frame.loc[frame[LABEL_COLUMN] == label, AMOUNT_PAID_COLUMN]
    ax.hist(subset.clip(upper=15_000), bins=60, color='#0f172a' if label == 0 else '#ef4444')
    ax.axvline(10_000, color='#f59e0b', linestyle='--', label='CTR threshold')
    ax.set_title(f'Class={label} ({"licit" if label == 0 else "illicit"})')
    ax.set_xlabel('Amount paid (USD, clipped at 15k)')
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Categorical feature signal

Per-category positive rate for `payment_format` and the top receiving currencies. Categories whose positive rate deviates meaningfully from the global rate carry signal and should survive into the model's one-hot encoding.

In [ ]:
for col in [PAYMENT_FORMAT_COLUMN, RECEIVING_CURRENCY_COLUMN]:
    pivot = (
        frame.groupby(col)[LABEL_COLUMN]
        .agg(['count', 'mean'])
        .sort_values('count', ascending=False)
        .head(10)
        .rename(columns={'count': 'n', 'mean': 'positive_rate'})
    )
    print(f'\nTop-10 by volume for {col}:')
    print(pivot)

## Takeaways carried forward

1. The class imbalance (~1000:1) dictates `scale_pos_weight` and `class_weight='balanced'` defaults in `configs/model_config.yaml`.
2. The amount bimodality at the CTR threshold validates the sub-threshold-share features in `src.features.entity_features`.
3. Per-category positive-rate gaps for payment_format support keeping it in the one-hot encoded feature set.